In [47]:
#conference
import json
import csv
from rdflib import Graph, RDF, URIRef, Literal, Namespace, OWL

RPO  = Namespace("http://www.semanticweb.org/ftsdemo/ontologies/2025/5/rpo#")
CSKG = Namespace("https://w3id.org/cskg/resource/")
CSKG_ONT = Namespace("https://w3id.org/cskg/ontology#")
PROV = Namespace("http://www.w3.org/ns/prov#")
FOAF = Namespace("http://xmlns.com/foaf/0.1/")

g = Graph()
g.bind("rpo", RPO)
g.bind("cskg", CSKG)
g.bind("cskg-ont", CSKG_ONT)
g.bind("prov", PROV)
g.bind("foaf", FOAF)
g.parse("full_trial_cleaned4.ttl", format = "turtle")

with open("articles_conferece.csv", newline="", encoding="utf-8") as fh:
    doc = csv.DictReader(fh)

    for row in doc:
        index = int(row["index"])
        conference = row["conf"]
        paper_uri = RPO[f"paper{index}"]

        if conference and paper_uri:
            conference_uri = RPO[conference.replace(" ", "")]
            g.add((paper_uri, RPO.published_at, conference_uri))
            g.add((conference_uri, RDF.type, RPO.Conference))
            g.add((conference_uri, RPO.name, Literal(conference)))


In [48]:
# ---------------------------- HELPERS ---------------------------
def strip_prefix(uri):
    u = str(uri)
    if "#" in u:
        return u.split("#")[-1]
    return u.split("/")[-1]

In [49]:
counter = 1
for s, p, o in g.triples((None, RDF.type, None)):
    if o == RPO.Country or o == RPO.City:
        print(counter, strip_prefix(s))
        counter += 1

1 Aachen
2 Aalborg
3 Abingdon
4 Adelaide
5 Albstadt
6 AnnArbor
7 Armonk
8 Basingstoke
9 Bayreuth
10 BeloHorizonte
11 Bengaluru
12 Bergamo
13 Bergen
14 Bezons
15 Bochum
16 Boulder
17 Braunschweig
18 Brescia
19 Bresso
20 Brighton
21 Buccino
22 Buffalo
23 Cagliari
24 CapeTown
25 Cardiff
26 Cergy
27 Charlottesville
28 ChasseneuilduPoitou
29 Chengdu
30 Chennai
31 Chertsey
32 Chongqing
33 CiudadVictoria
34 ClermontFerrand
35 CoralGables
36 DaNang
37 Dalian
38 Darmstadt
39 Delmont
40 Depok
41 Dongguan
42 Dsseldorf
43 Durham
44 EastLansing
45 Edmonton
46 EggensteinLeopoldshafen
47 Egham
48 Eschborn
49 Espoo
50 Evanston
51 Exeter
52 Falmouth
53 FarmingtonHills
54 Fisciano
55 Flagstaff
56 Fontainebleau
57 Fornebu
58 Gandhinagar
59 GreaterNoida
60 Gttingen
61 Gvle
62 Halifax
63 Hamilton
64 Hanoi
65 Hengyang
66 Herzogenaurach
67 Hobart
68 Huddersfield
69 Ioannina
70 Istanbul
71 Jeddah
72 Jlich
73 Jnkping
74 Komsomolskiy
75 Krefeld
76 Kthen
77 Lausanne
78 Lbeck
79 LeChesnayRocquencourt
80 Leeds
81 

In [50]:
counter = 1
for s, p, o in g.triples((None, RPO.part_of, None)):
    for subj, pred, obj in g.triples((o, RDF.type, None)):
        if obj == RPO.Country or o == RPO.City or o == RPO.Continent:
            print(counter, "subj", strip_prefix(s), "part_of", strip_prefix(o))
            counter += 1

##### Geographical information is clean. Let's focus on classifying the remaining NER instances (now saved as RPO.Concept):

In [51]:
for s, p, o in g.triples((None, RDF.type, None)):
    if o == RPO.Concept:
        for subj, pred, obj in g.triples((s, RPO.part_of, None)):
            print(strip_prefix(s), "          type undefined (saved as concept), part_of", strip_prefix(obj))

ABloomFilter           type undefined (saved as concept), part_of bloom_filter
AComparativeStudyOfStreamReasoningEngines           type undefined (saved as concept), part_of stream_reasoning
AControlledCrowdsourcingApproachForPracticalOntologyExtensions241           type undefined (saved as concept), part_of controlled_crowdsourcing_approach
AControlledCrowdsourcingApproachForPracticalOntologyExtensions243           type undefined (saved as concept), part_of controlled_crowdsourcing_approach
AConvolutionalNeuralNetwork           type undefined (saved as concept), part_of convolutional_neural_network
ADssSpringerInternationalPublishingSwitzerland           type undefined (saved as concept), part_of Springer_International_Publishing
AFrameKnowledgeRepresentationSystem           type undefined (saved as concept), part_of knowledge_representation
AFrameworkForQualityAssessmentOfAnnotations           type undefined (saved as concept), part_of quality
AGeneratingSemanticAspectsForQueries169 

In [52]:
import requests
import time
import json
import re

def split_cap_sequences(s):
    s = re.sub(r'^\d+|\d+$', '', s)
    if s.isupper():
        return s

    def split_caps(match):
        seq = match.group(0)
        return seq[:-1] + ' ' + seq[-1]

    s = re.sub(r'[A-Z]{2,}(?=[A-Z][a-z]|[a-z])', split_caps, s)
    s = re.sub(r'(?<=[a-z])([A-Z])', r' \1', s)
    return re.sub(r'\s+', ' ', s).strip()

def camel_to_readable(s):
    s = re.sub(r'^\d+|\d+$', '', s)
    if s.isupper():
        return s
    s = split_cap_sequences(s)

    out = []
    for i, c in enumerate(s):
        if c.isupper() and i > 0 and not s[i-1].isupper():
            out.append(" ")
        out.append(c)
    return re.sub(r'\s+', ' ', ''.join(out))

def get_wikidata_types(entities):
    results = {}
    for entity in entities:
        search_term = camel_to_readable(entity)
        #print(f"Processing: {entity}  -> searching as '{search_term}'")
        try:
            search_url = "https://www.wikidata.org/w/api.php"
            params = {
                'action': 'wbsearchentities',
                'search': search_term,
                'language': 'en',
                'format': 'json',
                'limit': 1
            }
            headers = {
                'User-Agent': 'Mozilla/5.0 (compatible; WikidataTypeFinder/1.0)'
            }
            response = requests.get(search_url, params=params, headers=headers, timeout=15)
            data = response.json()
            if not data.get('search'):
                search_term_underscore = re.sub(r"(the |for |of |and |dela |de )", " ", search_term)
                #print(f"searching {search_term_underscore}")
                params['search'] = search_term_underscore
                response = requests.get(search_url, params=params, headers=headers, timeout=15)
                data = response.json()
                if not data.get('search'):
                    results[entity] = {'types': [], 'relations': {}}
                    continue

            entity_id = data['search'][0]['id']
            entity_label = data['search'][0].get('label', 'No label')
            print(f"  Found: {entity_label} (ID: {entity_id})")

            sparql_query = f"""
            SELECT ?type ?typeLabel ?partOf ?partOfLabel ?ownedBy ?ownedByLabel ?keyPerson ?keyPersonLabel ?ceo ?ceoLabel ?loc ?locLabel WHERE {{
              wd:{entity_id} wdt:P31 ?type .
              OPTIONAL {{ wd:{entity_id} wdt:P361 ?partOf . }}
              OPTIONAL {{ wd:{entity_id} wdt:P127 ?ownedBy . }}
              OPTIONAL {{ wd:{entity_id} wdt:P3342 ?keyPerson . }}
              OPTIONAL {{ wd:{entity_id} wdt:P169 ?ceo . }}
              OPTIONAL {{ wd:{entity_id} wdt:P17 ?loc . }}
              SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
            }}
            """

            sparql_url = "https://query.wikidata.org/sparql"
            sparql_params = {
                'format': 'json',
                'query': sparql_query
            }

            sparql_response = requests.get(sparql_url, params=sparql_params, headers=headers, timeout=15)
            sparql_data = sparql_response.json()
            bindings = sparql_data['results']['bindings']

            types = set()
            relations = {'part_of': set(), 'owned_by': set(), 'key_person': set(), 'ceo': set(), 'loc':set()}

            for b in bindings:
                if 'typeLabel' in b:
                    types.add(b['typeLabel']['value'])
                if 'partOfLabel' in b:
                    relations['part_of'].add(b['partOfLabel']['value'])
                if 'ownedByLabel' in b:
                    relations['owned_by'].add(b['ownedByLabel']['value'])
                if 'keyPersonLabel' in b:
                    relations['key_person'].add(b['keyPersonLabel']['value'])
                if 'ceoLabel' in b:
                    relations['ceo'].add(b['ceoLabel']['value'])
                if 'locLabel' in b:
                    relations['loc'].add(b['locLabel']['value'])

            results[entity] = {
                'wikidata_label': entity_label,
                'types': list(types),
                'relations': {k: list(v) for k,v in relations.items()}
            }

            print(f"  types: {results[entity]['types']}")
            print(f"  relations: {results[entity]['relations']}")
        except Exception as e:
            print(f"  Error: {str(e)}")
            results[entity] = {'types': [], 'relations': {}}

        time.sleep(0.3)

    return results

In [53]:
companies_to_check = []
checked = []

for s, p, o in g.triples((None, RDF.type, None)):
    if o == RPO.Company and strip_prefix(s).lower() not in checked:
        companies_to_check.append(strip_prefix(s))
        checked.append(strip_prefix(s).lower())

results = get_wikidata_types(companies_to_check)

  Found: DNV (ID: Q18455539)
  types: ['classification society', 'business']
  relations: {'part_of': [], 'owned_by': ['Det Norske Veritas', 'Det Norske Veritas Foundation'], 'key_person': [], 'ceo': ['Remi Eriksen'], 'loc': ['Norway']}
  Found: Fluid Operations (ID: Q17085780)
  types: ['business']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': ['Germany']}
  Found: Huawei (ID: Q160120)
  types: ['enterprise', 'technology company', 'business']
  relations: {'part_of': [], 'owned_by': ['Ren Zhengfei'], 'key_person': [], 'ceo': ['Ren Zhengfei'], 'loc': ["People's Republic of China"]}
  Found: Iberia (ID: Q187972)
  types: ['airline']
  relations: {'part_of': [], 'owned_by': ['International Airlines Group'], 'key_person': [], 'ceo': ['Antonio Vázquez Romero'], 'loc': ['Spain']}
  Found: Meizu (ID: Q1143917)
  types: ['enterprise', 'business']
  relations: {'part_of': [], 'owned_by': ['Geely'], 'key_person': [], 'ceo': [], 'loc': ["People's Republic of Ch

In [54]:
#MAP
CATEGORY_KEYWORDS = {
    "Company": ["enterprise", "corporation", "business", "company", "manufacturer", "conglomerate"],
    "StateInstitution": ["minist", "federal", "govern", "authority"],
    "InternationalOrganisation": ["international", "intergov", "supranation", "global", "European"],
    "PoliticalOrganisation": ["politic", "party", "legis", "parliament"],
    "NGO": ["ngo", "nonprofit organization", "foundation", "voluntar", "labor", "union"],
    "University": ["university", "school", "academy", "education", "polytech"],
    "MilitaryOrganisation": ["milit", "army", "navy", "airforce", "guerrilla"],
    "PublishingCompany": ["publish", "press", "publisher"],
    "Person": ["human", "person", "individual"],
    "Organisation": ["research", "laborator", "scientific", "science", "institut", "bank"],
    "OtherEntity": ["data", "software", "system"],
    "Funder": ["funder", "funding", "grant"],
    "City": ["municipality"]
}

countries = [
    "afghanistan","albania","algeria","andorra","angola","antiguaanbarbuda","argentina",
    "armenia","australia","austria","azerbaijan","bahamas","bahrain","bangladesh","barbados",
    "belarus","belgium","belize","benin","bhutan","bolivia","bosniaandherzegovina",
    "botswana","brazil","brunei","bulgaria","burkinafaso","burundi","caboverde","cambodia",
    "cameroon","canada","centralafricanrepublic","chad","chile","china","colombia","comoros",
    "congo","costarica","croatia","cuba","cyprus","czechrepublic","democraticrepublicofthecongo",
    "denmark","djibouti","dominica","dominicanrepublic","ecuador","egypt","elsalvador",
    "equatorialguinea","eritrea","estonia","eswatini","ethiopia","fiji","finland","france",
    "gabon","gambia","georgia","germany","ghana","greece","grenada","guatemala","guinea",
    "guineabissau","guyana","haiti","honduras","hungary","iceland","india","indonesia",
    "iran","iraq","ireland","israel","italy","jamaica","japan","jordan","kazakhstan",
    "kenya","kiribati","kuwait","kyrgyzstan","laos","latvia","lebanon","lesotho","liberia",
    "libya","liechtenstein","lithuania","luxembourg","madagascar","malawi","malaysia",
    "maldives","mali","malta","marshallislands","mauritania","mauritius","mexico",
    "micronesia","moldova","monaco","mongolia","montenegro","morocco","mozambique","myanmar",
    "namibia","nauru","nepal","netherlands","newzealand","nicaragua","niger","nigeria",
    "northkorea","northmacedonia","norway","oman","pakistan","palau","palestine", "panama",
    "papuanewguinea","paraguay","peru","philippines","poland","portugal","qatar","romania",
    "russia","rwanda","saintkittsandnevis","saintlucia","saintvincentandthegrenadines",
    "samoa","sanmarino","saotomeandprincipe","saudiarabia","senegal","serbia","seychelles",
    "sierraleone","singapore","slovakia","slovenia","solomonislands","somalia","southafrica",
    "southkorea","southsudan","spain","srilanka","sudan","suriname","sweden","switzerland",
    "syria","taiwan","tajikistan","tanzania","thailand","timorleste","togo","tonga",
    "trinidadandtobago","tunisia","turkey","turkmenistan","tuvalu","uganda","ukraine",
    "unitedarabemirates","unitedkingdom","unitedstates","uruguay","uzbekistan","vanuatu",
    "vaticancity","venezuela","vietnam","yemen","zambia","zimbabwe"
]

continents = ["europe", "asia", "northamerica", "southamerica", "australia", "africa"]

def to_camel(s):
    s = re.sub(r'[^0-9a-zA-Z]+', ' ', s)
    parts = s.split()
    return ''.join(w.capitalize() for w in parts)



In [55]:
from rdflib import RDFS

for ent, info in results.items():
    entity_name = ent.lower()
    ent_uri = RPO[ent]
    g.add((ent_uri, RDFS.label, Literal(entity_name, lang="en")))

    for loc in countries + continents:
        if loc in entity_name:
            g.add((ent_uri, RPO.related_to, RPO[loc.title()]))

    relations = info.get("relations", {})

    if not relations:
        print(f"     ============== {entity_name} ============== no relations")
        continue

    for superpart in relations.get("part_of", []):
        g.add((ent_uri, RPO.related_to, RPO[to_camel(superpart)]))
        g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
        print(f"     added {entity_name} part_of {superpart}")

    for superpart in relations.get("owned_by", []):
        g.add((ent_uri, RPO.owned_by, RPO[to_camel(superpart)]))
        g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
        print(f"     added {entity_name} owned_by {superpart}")

    for superpart in relations.get("key_person", []):
        g.add((ent_uri, RPO.key_person, RPO[to_camel(superpart)]))
        g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
        g.add((RPO[to_camel(superpart)], RDF.type, RPO.Person))
        print(f"     added {entity_name} key_person {superpart}")

    for superpart in relations.get("ceo", []):
        g.add((ent_uri, RPO.ceo, RPO[to_camel(superpart)]))
        g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
        g.add((RPO[to_camel(superpart)], RDF.type, RPO.IndustryOwner))
        print(f"     added {entity_name} ceo {superpart}")

    if relations.get("loc"):
        loc = relations["loc"][0]
        g.add((ent_uri, RPO.located_in, RPO[to_camel(loc)]))


     added dnv owned_by Det Norske Veritas
     added dnv owned_by Det Norske Veritas Foundation
     added dnv ceo Remi Eriksen
     ============== digilensunitedstates ============== no relations
     ============== flaviait ============== no relations
     added huaweitechnologies owned_by Ren Zhengfei
     added huaweitechnologies ceo Ren Zhengfei
     added iberia owned_by International Airlines Group
     added iberia ceo Antonio Vázquez Romero
     ============== institutfrbiomedizinischeanalytikundnmrimaging ============== no relations
     added meizu owned_by Geely
     ============== ricercasulsistemaenergetico ============== no relations
     ============== ricercasulsistemaenergeticoitaly ============== no relations
     added siemenshealthcare part_of DAX
     added siemenshealthcare part_of MDAX
     added siemenshealthcare part_of CDAX
     added siemenshealthcare part_of TecDAX
     added siemenshealthcare ceo Bernd Montag
     ============== systemsapplicationsproduct

In [ ]:
#a little bit more on companies
to_check_dict = {}
to_check_lower = []

for s, p, o in g.triples((None, RDF.type, None)):
    if o in [RPO.Concept, RPO.Fund] and strip_prefix(s).lower() not in checked:
        to_check_dict[strip_prefix(s)] = s
        checked.append(strip_prefix(s).lower())

results = get_wikidata_types(list(to_check_dict.keys()))
    
print("\n" + "="*60)
print("FINAL RESULTS:")
print("="*60)
for entity, info in results.items():
    print(f"{entity}:")
    if 'wikidata_label' in info:
        print(f"  Wikidata label: {info['wikidata_label']}")
    #print(f"  Type: {info['types']}")
    #print(f"  relations: {info['relations']}")
    #print()

  Found: alpha-1-antitrypsin deficiency (ID: Q622437)
  types: ['class of disease', 'rare disease', 'designated intractable/rare disease', 'symptom or sign']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: SNTB1 (ID: Q18031637)
  types: ['gene']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: Ca-alpha1D (ID: Q29720984)
  types: ['gene']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: ATXN2L (ID: Q18036448)
  types: ['gene']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: AAA (ID: Q280732)
  types: ['musical group']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: AAAI Conference on Artificial Intelligence (ID: Q56682083)
  types: ['academic conference series']
  relations: {'part_of': [], 'owned_by': [], 'key_person': [], 'ceo': [], 'loc': []}
  Found: Aachen (

In [62]:
print(checked)

['dnv', 'digilensunitedstates', 'flaviait', 'fluidoperations', 'huaweitechnologies', 'iberia', 'institutfrbiomedizinischeanalytikundnmrimaging', 'meizu', 'ricercasulsistemaenergetico', 'ricercasulsistemaenergeticoitaly', 'roche', 'schaeffler', 'siemenshealthcare', 'systemsapplicationsproductsindataprocessing', 'total', 'zambon', 'bmwgroup', 'caprarotechnologiesunitedstates', 'contextualchangeunitedstates', 'datev', 'datevgermany', 'equinor', 'evidencebasedresearchunitedstates', 'ferrari', 'flaviaitgermany', 'flandersmake', 'fluidoperationsgermany', 'huaweitechnologiesunitedkingdom', 'ibmresearchchina', 'japanlifelinejapan', 'meizuchina', 'mondeca', 'mondecafrance', 'ntt', 'oxfordtechnologiesunitedkingdom', 'querydecomposer', 'robertbosch', 'robertboschunitedstates', 'siemenshealthcaregermany', 'siemensunitedstates', 'socitnationaledescheminsdeferfranais', 'socitnationaledescheminsdeferfranaisfrance', 'terumobelgium', 'virtualvehicle', 'virtualvehicleaustria', 'vodafonespain', 'volkswag

In [64]:
print(len(results))

15104


In [65]:
for ent, info in results.items():
    entity_name = ent.lower()
    for keyword in CATEGORY_KEYWORDS.keys():
        keywords = CATEGORY_KEYWORDS[keyword]
        if any(kw in t for kw in keywords for t in info["types"]):
            ent_uri = RPO[ent]
            for s, p, o in g.triples((ent_uri, RDF.type, None)):
                if o != RPO.Fund:
                	g.remove((ent_uri, RDF.type, o))
                
            g.add((ent_uri, RDF.type, RPO[keyword]))
            g.add((ent_uri, RDFS.label, Literal(entity_name, lang="en")))
            
            for loc in countries+continents:
                if loc in entity_name.lower():
                    g.add((ent_uri, RPO.related_to, RPO[loc.title()]))   
            print(f"Update {entity_name} -> {keyword}")

            if len(info["relations"]["part_of"]) > 0:
                print(" ========",entity_name ,"================= PART OF ==========================", keyword)
                for superpart in info["relations"]["part_of"]:
                    g.add((ent_uri, RPO.part_of, RPO[to_camel(superpart)]))
                    g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                print(f"     added {entity_name} part_of {superpart}")
            if len(info["relations"]["owned_by"]) > 0:
                for superpart in info["relations"]["owned_by"]:
                    g.add((ent_uri, RPO.owned_by, RPO[to_camel(superpart)]))
                    g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                print(f"     added {entity_name} owned_by {superpart}")
            if len(info["relations"]["key_person"]) > 0:
                for superpart in info["relations"]["key_person"]:
                    g.add((ent_uri, RPO.key_person, RPO[to_camel(superpart)]))
                    g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                    g.add((RPO[to_camel(superpart)], RDF.type, RPO.Person))
                print(f"     added {entity_name} key_person {superpart}")
            if len(info["relations"]["ceo"]) > 0:
                for superpart in info["relations"]["ceo"]:
                    g.add((ent_uri, RPO.ceo, RPO[to_camel(superpart)]))
                    g.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                    g.add((RPO[to_camel(superpart)], RDF.type, RPO.IndustryOwner))
                print(f"     added {entity_name} ceo {superpart}")
            if len(info["relations"]["loc"]) > 0:
                g.add((ent_uri, RPO.located_in, RPO[to_camel(info["relations"]["loc"][0])]))
                print(f"     added location")
            break

Update aac -> City
 ======== aac ================= PART OF ========================== City
     added aac part_of Rhineland
     added location
Update aace -> NGO
     added location
Update aaup5 -> NGO
     added location
Update abn -> Company
     added abn owned_by Kingdom of the Netherlands
     added abn ceo Robert Swaak
     added location
Update abstat -> City
 ======== abstat ================= PART OF ========================== City
     added abstat part_of Schozach-Bottwartal GVV
     added location
Update acm -> Company
     added location
Update adms -> Organisation
     added location
Update adms34 -> Organisation
     added location
Update adw -> University
     added location
Update aemet -> StateInstitution
     added location
Update aeris -> Person
Update aes -> Company
 ======== aes ================= PART OF ========================== Company
     added aes part_of S&P 500
     added aes ceo Andrés R. Gluski
     added location
Update afrl -> NGO
     added location
U

In [66]:
#where countries are located (continents)
def assign_countries_to_continents(graph):
    mapping = {
        "europe": [
            "albania","andorra","austria","belarus","belgium","bosniaandherzegovina",
            "bulgaria","croatia","cyprus","czechrepublic","denmark","estonia","finland",
            "france","germany","greece","hungary","iceland","ireland","italy","latvia",
            "liechtenstein","lithuania","luxembourg","malta","moldova","monaco","montenegro",
            "netherlands","northmacedonia","norway","poland","portugal","romania","russia",
            "sanmarino","serbia","slovakia","slovenia","spain","sweden","switzerland",
            "ukraine","unitedkingdom","vaticancity"
        ],
        "asia": [
            "afghanistan","armenia","azerbaijan","bahrain","bangladesh","bhutan","brunei",
            "cambodia","china","cyprus","georgia","india","indonesia","iran","iraq","israel",
            "japan","jordan","kazakhstan","kuwait","kyrgyzstan","laos","lebanon","malaysia",
            "maldives","mongolia","myanmar","nepal","northkorea","oman","pakistan","palestine",
            "philippines","qatar","saudiarabia","singapore","southkorea","srilanka","syria",
            "taiwan","tajikistan","thailand","timorleste","turkey","turkmenistan","unitedarabemirates",
            "uzbekistan","vietnam","yemen"
        ],
        "africa": [
            "algeria","angola","benin","botswana","burkinafaso","burundi","caboverde",
            "cameroon","centralafricanrepublic","chad","comoros","congo","democraticrepublicofthecongo",
            "djibouti","egypt","equatorialguinea","eritrea","eswatini","ethiopia","gabon","gambia",
            "ghana","guinea","guineabissau","ivorycoast","kenya","lesotho","liberia","libya",
            "madagascar","malawi","mali","mauritania","mauritius","morocco","mozambique","namibia",
            "niger","nigeria","rwanda","saotomeandprincipe","senegal","seychelles","sierraleone",
            "somalia","southafrica","southsudan","sudan","tanzania","togo","tunisia","uganda",
            "zambia","zimbabwe"
        ],
        "northamerica": [
            "antiguaanbarbuda","bahamas","barbados","belize","canada","costarica","cuba",
            "dominica","dominicanrepublic","elsalvador","grenada","guatemala","haiti","honduras",
            "jamaica","mexico","nicaragua","panama","saintkittsandnevis","saintlucia",
            "saintvincentandthegrenadines","trinidadandtobago","unitedstates"
        ],
        "southamerica": [
            "argentina","bolivia","brazil","chile","colombia","ecuador","guyana","paraguay",
            "peru","suriname","uruguay","venezuela"
        ],
        "australia": [
            "australia","fiji","kiribati","marshallislands","micronesia","nauru","newzealand",
            "palau","papuanewguinea","samoa","solomonislands","tonga","tuvalu","vanuatu"
        ]
    }

    country_to_continent = {c: cont for cont, clist in mapping.items() for c in clist}

    for country_uri in graph.subjects(RDF.type, RPO.Country):
        country_name = str(country_uri).split("#")[-1].lower()
        if country_name in country_to_continent:
            continent = country_to_continent[country_name]
            graph.add((country_uri, RPO.located_in, RPO[continent]))
            
assign_countries_to_continents(g)

In [67]:
def more_numbers(s: str) -> bool:
    if len(s) <= 4:
        return False
    digits = sum(c.isdigit() for c in s)
    letters = sum(c.isalpha() for c in s)
    return digits > letters

for fund in g.subjects(RDF.type, RPO.Fund):
    ent_types = []
    for ent_type in g.objects(fund, RDF.type):
        ent_types.append(ent_type)
    if not any(tp in ent_types for tp in [RPO.Organisation, RPO.Company, RPO.University, RPO.NGO, RPO.StateInstitution, RPO.MilitaryOrganisation, RPO.PoliticalOrganisation]):
        print("entity is only funder")
        if more_numbers(strip_prefix(fund)):
            print(f'{strip_prefix(fund)} added as grant')
            g.add((fund, RDF.type, RPO.Grant))
        else:
            g.remove((fund, RDF.type, RPO.Fund))
            g.add((fund, RDF.type, RPO.Concept))
            for paper_uri in g.subjects(RPO.funded_by, fund):
                g.remove((paper_uri, RPO.funded_by, fund))
                g.add((paper_uri, RPO.mentions, fund))

entity is only funder
CE480019 added as grant
entity is only funder
Ga318159 added as grant
entity is only funder
ICT316404 added as grant
entity is only funder
IIS1351029 added as grant
entity is only funder
IS21084 added as grant
entity is only funder
QE2114B added as grant
entity is only funder
TIN201678011 added as grant
entity is only funder
U01HG008473 added as grant
entity is only funder
U20A20174 added as grant


In [68]:
g.serialize("RPKG.ttl", format = "turtle")

<Graph identifier=N5a71c1f75bec45a58c1479e2863fed76 (<class 'rdflib.graph.Graph'>)>

In [70]:
dupl = g

for s, p, o in dupl.triples((None, RPO.observed_in_sentence, None)):
    ob = strip_prefix(o)
    if "/" in o:
        dupl.remove((s, p, o))
        dupl.add((s, p, RPO[ob.replace("/", "")]))
        print(ob.replace("/", ""))

paper752sent3533
paper370sent2034
paper370sent2034
paper370sent2034
paper733sent6232
paper387sent1198
paper387sent1198
paper387sent1198
paper387sent1431
paper387sent409
paper387sent409
paper387sent4571
paper73sent2356
paper77sent671
paper77sent671
paper242sent4261
paper268sent1189
paper268sent1189
paper268sent1189
paper268sent1189
paper268sent1189
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent3210
paper960sent9397
paper606sent106
paper606sent106
paper606sent106
paper606sent106
paper606sent1485
paper606sent1485
paper606sent1606
paper606sent1606
paper606sent2005
paper606sent2074
paper606sent2074
paper606sent2083
paper606sent2083
paper606sent2447
paper606sent2447
paper606sent2447
paper606sent2765
paper606sent535
paper606sent5372
paper606sent5372
paper606sent5372
paper606sent5907
paper606sent6051
paper606sent6940
paper1015sent1314
paper1015sent1314
paper1015sent1314
paper102sent0
paper102sent0
paper102sent

In [78]:
newdict = {}

for s, p, o in g.triples((None, RDF.type, None)):
    if o in [RPO.University, RPO.NGO, RPO.StateInstitution, RPO.MilitaryOrganisation, RPO.InternationalOrganisation, RPO.City]:
        if RPO.located_in not in g.predicates(s, None):
            print(strip_prefix(o))
            newdict[strip_prefix(s)] = s
        #checked.append(strip_prefix(s).lower())

results_uni = get_wikidata_types(list(newdict.keys()))

University
University
University
University
University
University
University
University
University
University
University
City
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
MilitaryOrganisation
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
NGO
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
StateInstitution
InternationalOrganisation
InternationalOrganisation
InternationalOrgani

In [79]:
for ent, info in results_uni.items():
    entity_name = ent.lower()
    for keyword in CATEGORY_KEYWORDS.keys():
        keywords = CATEGORY_KEYWORDS[keyword]
        if any(kw in t for kw in keywords for t in info["types"]):
            ent_uri = RPO[ent]
            for s, p, o in dupl.triples((ent_uri, RDF.type, None)):
                if o != RPO.Fund:
                	dupl.remove((ent_uri, RDF.type, o))
                
            dupl.add((ent_uri, RDF.type, RPO[keyword]))
            dupl.add((ent_uri, RDFS.label, Literal(entity_name, lang="en")))
            
            for loc in countries+continents:
                if loc in entity_name.lower():
                    dupl.add((ent_uri, RPO.related_to, RPO[loc.title()]))   
            print(f"Update {entity_name} -> {keyword}")

            if len(info["relations"]["part_of"]) > 0:
                print(" ========",entity_name ,"================= PART OF ==========================", keyword)
                for superpart in info["relations"]["part_of"]:
                    dupl.add((ent_uri, RPO.part_of, RPO[to_camel(superpart)]))
                    dupl.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                print(f"     added {entity_name} part_of {superpart}")
            if len(info["relations"]["owned_by"]) > 0:
                for superpart in info["relations"]["owned_by"]:
                    dupl.add((ent_uri, RPO.owned_by, RPO[to_camel(superpart)]))
                    dupl.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                print(f"     added {entity_name} owned_by {superpart}")
            if len(info["relations"]["key_person"]) > 0:
                for superpart in info["relations"]["key_person"]:
                    dupl.add((ent_uri, RPO.key_person, RPO[to_camel(superpart)]))
                    dupl.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                    dupl.add((RPO[to_camel(superpart)], RDF.type, RPO.Person))
                print(f"     added {entity_name} key_person {superpart}")
            if len(info["relations"]["ceo"]) > 0:
                for superpart in info["relations"]["ceo"]:
                    dupl.add((ent_uri, RPO.ceo, RPO[to_camel(superpart)]))
                    dupl.add((RPO[to_camel(superpart)], RDFS.label, Literal(superpart)))
                    dupl.add((RPO[to_camel(superpart)], RDF.type, RPO.IndustryOwner))
                print(f"     added {entity_name} ceo {superpart}")
            if len(info["relations"]["loc"]) > 0:
                dupl.add((ent_uri, RPO.located_in, RPO[to_camel(info["relations"]["loc"][0])]))
                print(f"     added location")
            break

Update commerce -> University
 ======== commerce ================= PART OF ========================== University
     added commerce part_of economics
Update doctorate -> University
Update educationalresources -> University
 ======== educationalresources ================= PART OF ========================== University
     added educationalresources part_of learning
Update hei -> University
Update iriu -> University
Update nyaya -> University
Update pragma -> University
Update technologyenhancedlearning -> University
Update universities3 -> University
Update university -> University
Update unitedstatesnavy -> MilitaryOrganisation
 ======== unitedstatesnavy ================= PART OF ========================== MilitaryOrganisation
     added unitedstatesnavy part_of United States Navy SEALs
     added location
Update afv -> MilitaryOrganisation
Update airforce -> MilitaryOrganisation
Update armyunit -> MilitaryOrganisation
Update cpt -> MilitaryOrganisation
Update command -> MilitaryOrgan

In [80]:
dupl.serialize("RPKG.ttl", format = "turtle")

<Graph identifier=N5a71c1f75bec45a58c1479e2863fed76 (<class 'rdflib.graph.Graph'>)>